<a href="https://colab.research.google.com/github/AdamClarkStandke/TransDiffTemp/blob/main/Chapter4_GPTmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coding An LLM Architecture

This notebook implements chapter 4 of the book [Build a Large Language Model (From Scratch) by Sebastian Raschka](https://www.amazon.com/Build-Large-Language-Model-Scratch/dp/1633437167) which builds the skeletal architecture for the most popular type of Large Language Model (LLM) right now, the Generative Pretrained Transformer (GPT). As the book details, as complex as GPT models seem, most of the architecture is repetative. Other than the initial components of the input embeddings and the final output layer, the main component of the GPT model is the **Transformer Component.** As the book details:

>  The transformer block, is a fundamental building block of GPT and other LLM architectures. This block, which is repeated a dozen times in the 124-million-parameter GPT-2 architecture, combines the components of: **multi-headed attention**, **layer normalization**, **dropout**, **feed-forward layers**, and **GELU activations**.

Multi-headed attention was coded in the notebook titled [Coding Attention](https://github.com/AdamClarkStandke/TransDiffTemp/blob/main/CodingAttention_ch3.ipynb) and will be imported from [here](https://github.com/AdamClarkStandke/TransDiffTemp/blob/main/previous_chapters.py). As stated above the configuration for this LLM Architecture is the 124 million parameter version of [GPT-2](https://github.com/openai/gpt-2?tab=readme-ov-file) developed by [OpenAI](https://openai.com/) which will be used in later chapters for training and instruction fine-tuning. The configuration for the model is found in the ```GPT_CONFIG_124M ``` dictionary and consists of the following model parameters:
*  **vocab_size**: refers to a vocabulary of 50,257 words, as used by the [Byte-Pair Encoding (BPE) tokenizer](https://github.com/openai/tiktoken) of GPT-2
*   **context_length**: denotes the maximum number of input tokens the model can handle via the [positional embeddings](https://towardsdatascience.com/positional-embeddings-in-transformers-a-math-guide-to-rope-alibi/)
*   **n_heads**: indicates the count of attention heads in the multi-headed attention mechanism
*  **n_layers**: specifies the number of transformer blocks in the model
*  **drop_rate**: indicates the intensity of the dropout mechanism to prevent overfitting
*   **qkv_bias**: determines whether to include a bias vector in the Linear Layers of the multi-headed attention for query, key, and value computations.






In [1]:
'''GPT-2 Config:'''
GPT_CONFIG_124M={
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [3]:
import torch
import torch.nn as nn
import previous_chapters

## Layer Normalization Component

Layer Normalization is used to stabilize the training of LLMs to avoid such training issues such as [vanishing or exploding gradients](https://en.wikipedia.org/wiki/Vanishing_gradient_problem). As the book details:
> The main idea behind layer normalization is to adjust the activations (outputs) of a neural network layer to have a **mean of 0 and a variance of 1**...[i]n GPT-2 and modern transformer architectures, **layer normalization is typically applied before and after multi-headed attention and before the final output layer**

The form of layer normalization used by the book consists of calculating the mean and variance of the un-normalized input tensor along its last dimension i.e., `dim=-1`. And then normalizing it by subtracting the mean from it and dividing by the square root of its variance and a small constant(for numerical stability).

Also two trainable parameters called `scale` and `shift` are applied to the normalized input tensor so that the LLM automatically adjusts these parameters during training for extra normalization!!! (Just in case).






In [4]:
class LayerNorm(torch.nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5 #avoid divide by zero constant
    self.scale = nn.Parameter(torch.ones(emb_dim)) #scale
    self.shift = nn.Parameter(torch.zeros(emb_dim)) #shift

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True) #mean calc
    var = x.var(dim=-1, keepdim=True, unbiased=False) #variance calc
    norm_x = (x-mean)/(torch.sqrt(var+self.eps)) #normalized output
    return self.scale * norm_x + self.shift #normalized output


## Feed-Forward Component

The Feed-Forward Network used in GPT-2 is composed of a neural network consisting of two Linear Layers and a Gaussian Error Linear Unit (GELU) as the activation function (more on that later).For GPT-2 the two Linear Layers are structured  to have the same shape for the input and output dimensions. Making the architecture easily stackable (since the input and output dimensions are the same).

Interestingly, while the input and output dimensions are the same the **hidden dimension is expanded rather than shrunken** like for [auto-encoders](https://en.wikipedia.org/wiki/Autoencoder). As the book details this architecture allows for better exploration (and not [exploitation](https://en.wikipedia.org/wiki/Exploration%E2%80%93exploitation_dilemma) 🤔) of the representation space:

> Although the input and output dimensions are the same, it internally **expands the embedding dimension** into a higher-dimensional space through the first linear layer...[t]his expansion is followed by a nonlinear GELU activation and then a contraction back to the original dimension with the second linear transformation. **Such a design allows for the exploration of a richer representation space**.

Accordingly the orginial embedding space of 768 dimensions is expanded by 4 times by the first layer to 3,072 dimensions. Then the second layer shrinks the space back to 768 dimensions.

The activation function that is applied after the first layer is a Gaussian Error Linear Unit (GeLU) rather than the typically used [Rectified Linear Unit (ReLU)](https://en.wikipedia.org/wiki/Rectified_linear_unit). GELU is a smooth activation function that outputs a **non-zero value for almost all negative input values** and is based on the [cumulative distribution function (cdf)](https://en.wikipedia.org/wiki/Cumulative_distribution_function) of the standard Gaussian distribution. This is different than the ReLU activation function which is a piecewise function that outputs **zero for all negative input values.** Which becomes a [problem](https://en.wikipedia.org/wiki/Vanishing_gradient_problem) when training very deep networks with the backpropagation algorithm.

As the book details the cdf function for GPT-2 was approximated and found via [curve fitting](https://en.wikipedia.org/wiki/Curve_fitting) to produce the following approximation of the GELU function:
$GELU(x)\approx 0.5\cdot x\cdot(1+tanh[\sqrt{\frac{2}{\pi}}\cdot(x+0.0044715\cdot x^3)])$




In [5]:
class GELU(torch.nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x): #GELU function
    return 0.5 * x *(1+torch.tanh(
        torch.sqrt(torch.tensor(2.0/torch.pi)) *
        (x + 0.044715 * torch.pow(x, 3))
    ))

class FeedForward(torch.nn.Module): #Feed-Forward NN
    def __init__(self, cfg):
      super().__init__()
      self.layers = nn.Sequential(
          nn.Linear(cfg['emb_dim'], 4*cfg['emb_dim']),
          GELU(),
          nn.Linear(4*cfg['emb_dim'], cfg['emb_dim'])
      )

    def forward(self, x):
      return self.layers(x)


## Transformer Component

As the name suggests the transformer is where all of the magic happens!! The transformer contains and **connects all of the previous components together; those being multi-headed attention, layer normalization, dropout, the feed-forward network and [shortcut/skip/residual connections](https://en.wikipedia.org/wiki/Residual_neural_network)**. Similarly, as previously mentioned when discussing the architecture of the feed-forward network, the transfomer maintains the same dimensions for the input and output tensors. This design not only allows the architecture to be easily **stackable** (ie., repeating the transfomer a million times 🍄) but more importantly, allows the transformer to be used across various [auto-regressive domains](https://deepgenerativemodels.github.io/notes/autoregressive/)(just look at this [library](https://huggingface.co/spaces/yonigozlan/Transformers-Timeline) for all the different transformer based architectures). As the book details:

> This design enables its effective application across a wide range of...tasks...**each output vector is a context vector that encapsulates information from the entire input sequence. So while the shape remains unchanged as it passes through the transformer, the content of each output vector is re-encoded to integrate contextual information from across the entire input sequence**

> The idea is that the **self-attention mechanism in the multi-head attention block identifies and analyzes relationships between elements in the input sequence. In contrast, the feed forward network modifies the data individually at each position**.






In [6]:
class TransformerBlock(torch.nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att = previous_chapters.MultiHeadAttention(
        d_in = cfg["emb_dim"],
        d_out = cfg['emb_dim'],
        context_length = cfg['context_length'],
        num_heads = cfg['n_heads'],
        dropout = cfg['drop_rate'],
        qkv_bias = cfg['qkv_bias']
    )
    self.ff = FeedForward(cfg)
    self.norm1 = LayerNorm(cfg['emb_dim'])
    self.norm2 = LayerNorm(cfg['emb_dim'])
    self.drop_shortcut = nn.Dropout(cfg['drop_rate'])

  def forward(self, x):
    shortcut = x  #initalize 1st skip-connect
    x = self.norm1(x) #Pre-layer Norm
    x = self.att(x) #Multi-headed Att
    x = self.drop_shortcut(x) #Dropout
    x = shortcut + x #Skip-connect

    shortcut = x  #initalize 2nd Skip-connect
    x = self.norm2(x) #Post-layer Norm
    x = self.ff(x) #Feed-Forward Network
    x = self.drop_shortcut(x) #Dropout
    x = shortcut + x #Skip-connect
    return x   #return context vec/tensor

## GPT-2 Model

Once all the building blocks have been created the main model can be assembled. As previously mentioned this notebook implements GPT-2 which was released by OpenAI to the open-source community back in 2019 and stems from the paper [Language Models are Unsupervised Multitask Learners authored by Alec Radford, Jeffrey Wu, Rewon Child,  David Luan, Dario Amodei, and Ilya Sutskever](https://github.com/openai/gpt-2)( back when OpenAI was still open 🤑). GPT-2 **stacks the transfomer block multiple times** (12 times to be exact) and houses the two other main components: the embedding layers i.e., **positional and token embedding layers** and the final output layer which is just a Linear Layer that outputs a 50,257 dimensional vector of logits; representing each token/word of the vocabulary. And that's it!!! All that has to be done now is train it!!




In [7]:
class GPT(torch.nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim']) #token emb
    self.pos_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim']) #pos emb
    self.drop_emb = nn.Dropout(cfg['drop_rate']) #dropout
    self.trf_blocks = nn.Sequential( #transformers
        *[TransformerBlock(cfg) for _ in range(cfg['n_layers'])]
    )
    self.final_norm = LayerNorm(cfg["emb_dim"]) #output norm
    self.out_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False) #output layer

  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(
        torch.arange(seq_len, device=in_idx.device)
    )
    x = pos_embeds + tok_embeds #combining pos and token embeds
    x = self.drop_emb(x) #pre-transformer dropout
    x = self.trf_blocks(x) #transformers
    x = self.final_norm(x) #output norm
    x = self.out_head(x) #output layer
    return x


## Generate Random Text

Eventhough the GPT-2 model has not been trained this section details the basics of the post-processing/text generation process. Just as there is a pre-processing step of encoding words into token IDs, there is a post-processing step that converts probabilties back into token IDs and then into words.

From a high-level perspective, tokens/words are generated in a sequential and iterative manner. First, GPT-2 generates the output tokens. Then **post-processing appends/concatenates these output tokens to the input context and feeds it back into GPT-2 to generate the output tokens.** This process is iterative and ends when the `max_new_tokens` value has been reached.

From a low-level perspective, GPT-2 outputs a three dimensional tensor of [logits](https://stackoverflow.com/questions/41455101/what-is-the-meaning-of-the-word-logits-in-tensorflow) with the shape of `[[[batch_size,num_token,vocab_size]]]` Then post-processing extracts the last row from the `num_token` dimension, which represents the next predicted word.Lastly, the logits are transformed into probabilities through the softmax function and the index with the highest probability value is the next predicted word.   

In [8]:
def generate_text_simple(model, idx, max_new_tokens, context_length):
  for _ in range(max_new_tokens):
    idx_cond = idx[:, -context_length:] #batched input context
    with torch.no_grad():
      logits = model(idx_cond) #input into model

    logits = logits[:, -1, :] #get last token
    probas = torch.softmax(logits, dim=-1) #calculate probs
    idx_next = torch.argmax(probas, dim=-1, keepdim=True) #get index w/ highest prob
    idx = torch.cat((idx, idx_next), dim=-1) #append word to input context
  return idx

In [10]:
#testing
start_context = "Hello, I am "
tokenizer = previous_chapters.tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(start_context)
#token IDs
print(f"Encoded Text: {encoded}")
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
model = GPT(GPT_CONFIG_124M)
model.eval()
out = generate_text_simple(
    model = model,
    idx = encoded_tensor,
    max_new_tokens = 6,
    context_length = GPT_CONFIG_124M["context_length"]
)
#weird stuff lol
print(f"Generated Text: {tokenizer.decode(out.squeeze(0).tolist())}")

Encoded Text: [15496, 11, 314, 716, 220]
Generated Text: Hello, I am  unbeliev catastrophic ;;waysProf nipple
